# I. ELASTIC-NET PIPELINE

## 1. CONFIG

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

DATA_PATH = "data/preprocessed_features.csv"
MODEL_DIR = "model/en_per_target_models"
RESULTS_PATH = "results/en_per_target_results.csv"

N_SPLITS = 5
RANDOM_STATE = 42

ALPHAS = np.logspace(-2, 0, 5)
L1_RATIOS = [0.2, 0.5, 0.8]

os.makedirs(MODEL_DIR, exist_ok=True)

## 2. LOAD DATA & SCALE INPUTS

In [ ]:
df = pd.read_csv(DATA_PATH)

user_col = "pass__user_id"
output_prefixes = (
    "num__liwc_",
    "num__topic_",
    "num__genius_",
    "num__music_track_",
)

output_cols = [c for c in df.columns if c.startswith(output_prefixes)]
input_cols = [c for c in df.columns if c not in output_cols]
input_cols.remove(user_col)

X = df[input_cols].values
groups = df[user_col].values

print(f"Samples: {len(X)}")
print(f"Inputs: {len(input_cols)}")
print(f"Targets: {len(output_cols)}")

# SCALE INPUTS
X_scaler = StandardScaler()
X_scaled = X_scaler.fit_transform(X)

## 3. USER-BLOCKED CV + GRID SEARCH

In [ ]:
cv = GroupKFold(n_splits=N_SPLITS)
rows = []

for target in output_cols:
    print(f"\nTraining ElasticNet for: {target}")

    y = df[target].values

    # Scale target (paper does this)
    y_scaler = StandardScaler()
    y_scaled = y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

    best_score = np.inf
    best_params = None

    # Hyperparameter search
    for alpha in ALPHAS:
        for l1 in L1_RATIOS:
            rmses = []

            for tr, te in cv.split(X_scaled, y_scaled, groups):
                model = ElasticNet(
                    alpha=alpha,
                    l1_ratio=l1,
                    max_iter=10000,
                    random_state=RANDOM_STATE
                )
                model.fit(X_scaled[tr], y_scaled[tr])

                preds = model.predict(X_scaled[te])
                rmse = np.sqrt(mean_squared_error(y_scaled[te], preds))
                rmses.append(rmse)

            mean_rmse = np.mean(rmses)

            if mean_rmse < best_score:
                best_score = mean_rmse
                best_params = (alpha, l1)

    # Final CV evaluation
    rmses, r2s = [], []

    for tr, te in cv.split(X_scaled, y_scaled, groups):
        model = ElasticNet(
            alpha=best_params[0],
            l1_ratio=best_params[1],
            max_iter=10000,
            random_state=RANDOM_STATE
        )
        model.fit(X_scaled[tr], y_scaled[tr])

        preds = model.predict(X_scaled[te])

        rmses.append(np.sqrt(mean_squared_error(y_scaled[te], preds)))
        r2s.append(r2_score(y_scaled[te], preds))

    rows.append({
        "target": target,
        "rmse_mean": np.mean(rmses),
        "rmse_std": np.std(rmses),
        "r2_mean": np.mean(r2s),
        "alpha": best_params[0],
        "l1_ratio": best_params[1]
    })

    # Train final model
    final_model = ElasticNet(
        alpha=best_params[0],
        l1_ratio=best_params[1],
        max_iter=10000,
        random_state=RANDOM_STATE
    )
    final_model.fit(X_scaled, y_scaled)

    joblib.dump(
        {
            "model": final_model,
            "X_scaler": X_scaler,
            "y_scaler": y_scaler,
            "input_cols": input_cols
        },
        os.path.join(MODEL_DIR, f"enet_{target}.joblib")
    )

In [ ]:
# SAVE RESULTS
results_df = pd.DataFrame(rows).sort_values("rmse_mean")
results_df.to_csv(RESULTS_PATH, index=False)

print("\n=== BEST TARGETS ===")
print(results_df.head(10))

print("\n=== WORST TARGETS ===")
print(results_df.tail(10))

# II. ELASTIC-NET INTERPRETATION

## 1. CONFIG

In [ ]:
MODEL_DIR = "model/en_per_target_models"
OUTPUT_CSV = "results/en_interpretation.csv"
OUTPUT_SUMMARY = "results/en_interpretation_summary.txt"

MIN_COEF_MAGNITUDE = 0.05  # below this, coefficient is considered "flat"
TOP_K_SUMMARY = 3          # number of features per direction in summary

## 2. ITERATE MODELS

In [ ]:
rows = []

for fname in os.listdir(MODEL_DIR):
    if not fname.startswith("enet_"):
        continue

    target = fname.replace("enet_", "").replace(".joblib", "")
    bundle = joblib.load(os.path.join(MODEL_DIR, fname))

    model = bundle["model"]
    features = bundle["input_cols"]
    coefs = model.coef_

    for f, c in zip(features, coefs):
        if abs(c) < MIN_COEF_MAGNITUDE:
            continue  # skip flat features
        direction = "positive" if c > 0 else "negative"

        rows.append({
            "target": target,
            "feature": f,
            "coef": c,
            "direction": direction,
            "magnitude": abs(c)
        })

## 3. CREATE OVERVIEW TABLE

In [ ]:
coef_df = pd.DataFrame(rows).sort_values(["target", "magnitude"], ascending=[True, False])
coef_df.to_csv(OUTPUT_CSV, index=False)
print("\nElastic-net interpretation saved to:", OUTPUT_CSV)

# HUMAN-READABLE SUMMARY
summary_rows = []
for target in coef_df["target"].unique():
    subset = coef_df[coef_df["target"] == target]
    pos = subset[subset["direction"] == "positive"]["feature"].tolist()
    neg = subset[subset["direction"] == "negative"]["feature"].tolist()

    statement = []
    if pos:
        statement.append(f"↑ {', '.join(pos[:TOP_K_SUMMARY])}")
    if neg:
        statement.append(f"↓ {', '.join(neg[:TOP_K_SUMMARY])}")

    summary_rows.append({"target": target, "summary": " | ".join(statement)})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_SUMMARY, index=False, header=False)
print(f"Readable summary saved to: {OUTPUT_SUMMARY}")
